# Merge & De-Duplication

In [1]:
import pandas as pd
import numpy as np
import torch
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import warnings
warnings.filterwarnings("ignore")

c:\Users\René S\Documents\ZHAW\Master\3. Semester\ARP\zhaw_arep\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import pandas as pd
import os

# File paths
file1 = 'english_news_v3.csv'
file2 = 'english_news2_raw.csv'

# Read both CSV files
print(f"Loading {file1}...")
df1 = pd.read_csv(file1)

print(f"Loading {file2}...")
df2 = pd.read_csv(file2)

# Print initial counts
print(f"\nInitial counts:")
print(f"  {file1}: {len(df1)} rows")
print(f"  {file2}: {len(df2)} rows")
print(f"  Total before merge: {len(df1) + len(df2)} rows")

# Merge the dataframes
df_merged = pd.concat([df1, df2], ignore_index=True)

# Remove duplicates based on URL (most reliable identifier)
df_merged = df_merged.drop_duplicates(subset=['url'], keep='first')

# Print summary
print(f"\nSummary:")
print(f"  Total rows after merge: {len(df_merged)}")
print(f"  Duplicates removed: {len(df1) + len(df2) - len(df_merged)}")
print(f"  Columns: {', '.join(df_merged.columns.tolist())}")
print(f"  Date range: {df_merged['publishedAt'].min()} to {df_merged['publishedAt'].max()}")



Loading english_news_v3.csv...
Loading english_news2_raw.csv...

Initial counts:
  english_news_v3.csv: 44379 rows
  english_news2_raw.csv: 61346 rows
  Total before merge: 105725 rows

Summary:
  Total rows after merge: 89517
  Duplicates removed: 16208
  Columns: publishedAt, title, source, description, url
  Date range: 2020-11-01 16:52:50+00:00 to 2025-10-31 12:48:33+00:00


In [2]:
# save the merged dataframe
df_merged.to_csv('english_news_v4.csv', index=False)

# First Topic Identification & Cleanup round

In [3]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import warnings
import numpy as np

warnings.filterwarnings('ignore')

print("BERTopic libraries imported successfully!")


/Users/nicolas/Desktop/Repos/zhaw_mle/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BERTopic libraries imported successfully!


In [4]:
# Load the merged dataset
df = pd.read_csv('english_news_v4.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['publishedAt'].min()} to {df['publishedAt'].max()}")

# Prepare text data for topic modeling
# Combine title and description, remove missing values
df_topic = df.copy()
df_topic['combined_text'] = df_topic.apply(
    lambda row: f"{row['title']} {row['description']}" 
    if pd.notna(row['title']) and pd.notna(row['description']) 
    else (row['title'] if pd.notna(row['title']) else str(row['description']) if pd.notna(row['description']) else ''),
    axis=1
)

# Remove empty strings and very short texts
df_topic = df_topic[df_topic['combined_text'].str.strip() != ''].copy()
df_topic = df_topic[df_topic['combined_text'].str.len() > 20].copy()  # Minimum length

print(f"\nArticles prepared for topic modeling: {len(df_topic)}")
print(f"Average document length: {np.mean([len(doc) for doc in df_topic['combined_text']]):.1f} characters")
print(f"\nSample text (first 200 chars):")
print(df_topic['combined_text'].iloc[0][:200] if len(df_topic) > 0 else "No data")


Dataset shape: (89517, 5)
Columns: ['publishedAt', 'title', 'source', 'description', 'url']

Date range: 2020-11-01 16:52:50+00:00 to 2025-10-31 12:48:33+00:00

Articles prepared for topic modeling: 89516
Average document length: 217.3 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.


In [5]:
# Extract documents for topic modeling
documents = df_topic['combined_text'].tolist()

print(f"Total documents: {len(documents)}")

# Initialize BERTopic with optimized settings for news articles
print("\nInitializing BERTopic model...")
print("This may take a few minutes...")

# Use a lightweight embedding model for faster processing
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Configure dimensionality reduction and clustering
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Initialize BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=10,
    verbose=True,
    language='english'
)

# Fit the model
print("\nFitting the model to documents...")
topics, probs = topic_model.fit_transform(documents)

print(f"\nModel training completed!")
print(f"Number of topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
if -1 in topics:
    print(f"Outliers (topic -1): {topics.count(-1)} documents")


Total documents: 89516

Initializing BERTopic model...
This may take a few minutes...


2025-10-31 14:27:55,949 - BERTopic - Embedding - Transforming documents to embeddings.



Fitting the model to documents...


Batches: 100%|██████████| 2798/2798 [01:18<00:00, 35.74it/s]
2025-10-31 14:29:15,296 - BERTopic - Embedding - Completed ✓
2025-10-31 14:29:15,296 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-31 14:30:21,809 - BERTopic - Dimensionality - Completed ✓
2025-10-31 14:30:21,811 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR


Model training completed!
Number of topics discovered: 265
Outliers (topic -1): 36075 documents


In [6]:
# Get topic information and display TOP 50 TOPICS
topic_info = topic_model.get_topic_info()

print("=" * 80)
print("TOP 50 TOPICS WITH KEYWORDS")
print("=" * 80)

# Filter out outlier topic (-1) for display
topics_to_show = topic_info[topic_info['Topic'] != -1].head(50)

print(f"\nTotal topics discovered (excluding outliers): {len(topic_info[topic_info['Topic'] != -1])}")
print(f"Documents assigned to topics: {topics_to_show['Count'].sum()}")
if -1 in topics:
    outliers_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if len(topic_info[topic_info['Topic'] == -1]) > 0 else 0
    print(f"Outliers (unassigned): {outliers_count}")

print("\n" + "=" * 80)
print(f"{'Rank':<6} {'Topic ID':<10} {'Articles':<12} {'Topic Name':<40} {'Top Keywords'}")
print("=" * 80)

for idx, (_, row) in enumerate(topics_to_show.iterrows(), 1):
    topic_id = row['Topic']
    count = row['Count']
    name = row['Name']
    
    # Get top words for this topic
    top_words = topic_model.get_topic(topic_id)
    words_str = ", ".join([word for word, score in top_words[:10]])
    
    # Shorten topic name if too long
    display_name = name[:37] + "..." if len(name) > 40 else name
    
    print(f"{idx:<6} {topic_id:<10} {count:<12} {display_name:<40} {words_str}")


TOP 50 TOPICS WITH KEYWORDS

Total topics discovered (excluding outliers): 265
Documents assigned to topics: 28714
Outliers (unassigned): 36075

Rank   Topic ID   Articles     Topic Name                               Top Keywords
1      0          3577         0_league_cup_england_champions           league, cup, england, champions, wales, rugby, manchester, final, euro, manager
2      1          1661         1_gaza_israel_hamas_israeli              gaza, israel, hamas, israeli, palestinian, netanyahu, palestinians, hostages, war, ceasefire
3      2          1355         2_electric_tesla_vehicles_car            electric, tesla, vehicles, car, ev, vehicle, cars, auto, battery, evs
4      3          1160         3_covid_19_vaccine_coronavirus           covid, 19, vaccine, coronavirus, pandemic, virus, cases, vaccines, variant, omicron
5      4          1123         4_swift_taylor_film_movie                swift, taylor, film, movie, her, oscars, star, show, music, netflix
6      5       

In [8]:
df_to_filter = df.copy()

# first set of words to filter out from headlines (case-insensitive)
final_words_to_filter = [
    'league', 'cup', 'champions', 'wales', 'rugby', 'manchester',
    'swift', 'taylor', 'film', 'movie', 'oscars', 'star', 'music', 'netflix',
    'transgender', 'students', 'trans', 'gender', 'dei',
    'immigration', 'deportation', 'deportations', 'migrants', 'immigrants',
    'nfl', 'draft', 'jaguars', 'broncos', 'raiders', 'panthers', 'vikings',
    'shooting', 'murder', 'killing', 'nba',
    
]

print("Final round of filtering...")
print(f"Current dataset size (from english_news_v2): {len(df_to_filter)} articles")

# Create a mask to identify articles with irrelevant headlines
# Check if title contains any of the final filter words (case-insensitive)
filter_mask_v1 = df_to_filter['title'].str.lower().str.contains('|'.join(final_words_to_filter), na=False, regex=True)

# Count articles to be removed
articles_to_remove_v1 = filter_mask_v1.sum()
print(f"Articles to remove in this round: {articles_to_remove_v1} ({articles_to_remove_v1/len(df_to_filter)*100:.2f}%)")

# Filter the dataframe (keep articles that DON'T match the filter)
df_filtered_v1 = df_to_filter[~filter_mask_v1].copy()

print(f"Filtered dataset size (v1): {len(df_filtered_v1)} articles")
print(f"Total articles removed in this round: {len(df_to_filter) - len(df_filtered_v1)}")
print(f"Total articles removed from original: {len(df) - len(df_filtered_v1)} ({(len(df) - len(df_filtered_v1))/len(df)*100:.2f}%)")

# Save the final filtered dataset
output_filename = 'english_news_v5.csv'

# Save to CSV (already has the correct columns)
df_filtered_v1.to_csv(output_filename, index=False)


Final round of filtering...
Current dataset size (from english_news_v2): 89517 articles
Articles to remove in this round: 7143 (7.98%)
Filtered dataset size (v1): 82374 articles
Total articles removed in this round: 7143
Total articles removed from original: 7143 (7.98%)

Sample of removed articles (this round):


# Second Topic Identification & Cleanup round

In [16]:
# Load the latest dataset
df = pd.read_csv('english_news_v5.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['publishedAt'].min()} to {df['publishedAt'].max()}")

# Prepare text data for topic modeling
# Combine title and description, remove missing values
df_topic = df.copy()
df_topic['combined_text'] = df_topic.apply(
    lambda row: f"{row['title']} {row['description']}" 
    if pd.notna(row['title']) and pd.notna(row['description']) 
    else (row['title'] if pd.notna(row['title']) else str(row['description']) if pd.notna(row['description']) else ''),
    axis=1
)

# Remove empty strings and very short texts
df_topic = df_topic[df_topic['combined_text'].str.strip() != ''].copy()
df_topic = df_topic[df_topic['combined_text'].str.len() > 20].copy()  # Minimum length

print(f"\nArticles prepared for topic modeling: {len(df_topic)}")
print(f"Average document length: {np.mean([len(doc) for doc in df_topic['combined_text']]):.1f} characters")
print(f"\nSample text (first 200 chars):")
print(df_topic['combined_text'].iloc[0][:200] if len(df_topic) > 0 else "No data")


Dataset shape: (82374, 5)
Columns: ['publishedAt', 'title', 'source', 'description', 'url']

Date range: 2020-11-01 16:52:50+00:00 to 2025-10-31 12:48:33+00:00

Articles prepared for topic modeling: 82373
Average document length: 217.4 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.


In [10]:
# Extract documents for topic modeling
documents = df_topic['combined_text'].tolist()

print(f"Total documents: {len(documents)}")

# Initialize BERTopic with optimized settings for news articles
print("\nInitializing BERTopic model...")
print("This may take a few minutes...")

# Use a lightweight embedding model for faster processing
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Configure dimensionality reduction and clustering
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Initialize BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=10,
    verbose=True,
    language='english'
)

# Fit the model
print("\nFitting the model to documents...")
topics, probs = topic_model.fit_transform(documents)

print(f"\nModel training completed!")
print(f"Number of topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
if -1 in topics:
    print(f"Outliers (topic -1): {topics.count(-1)} documents")


Total documents: 82373

Initializing BERTopic model...
This may take a few minutes...


2025-10-31 14:51:26,145 - BERTopic - Embedding - Transforming documents to embeddings.



Fitting the model to documents...


Batches: 100%|██████████| 2575/2575 [01:09<00:00, 36.98it/s]
2025-10-31 14:52:36,466 - BERTopic - Embedding - Completed ✓
2025-10-31 14:52:36,466 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-31 14:53:29,398 - BERTopic - Dimensionality - Completed ✓
2025-10-31 14:53:29,399 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR


Model training completed!
Number of topics discovered: 244
Outliers (topic -1): 33349 documents


In [11]:
# Get topic information and display TOP 50 TOPICS
topic_info = topic_model.get_topic_info()

print("=" * 80)
print("TOP 50 TOPICS WITH KEYWORDS")
print("=" * 80)

# Filter out outlier topic (-1) for display
topics_to_show = topic_info[topic_info['Topic'] != -1].head(50)

print(f"\nTotal topics discovered (excluding outliers): {len(topic_info[topic_info['Topic'] != -1])}")
print(f"Documents assigned to topics: {topics_to_show['Count'].sum()}")
if -1 in topics:
    outliers_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if len(topic_info[topic_info['Topic'] == -1]) > 0 else 0
    print(f"Outliers (unassigned): {outliers_count}")

print("\n" + "=" * 80)
print(f"{'Rank':<6} {'Topic ID':<10} {'Articles':<12} {'Topic Name':<40} {'Top Keywords'}")
print("=" * 80)

for idx, (_, row) in enumerate(topics_to_show.iterrows(), 1):
    topic_id = row['Topic']
    count = row['Count']
    name = row['Name']
    
    # Get top words for this topic
    top_words = topic_model.get_topic(topic_id)
    words_str = ", ".join([word for word, score in top_words[:10]])
    
    # Shorten topic name if too long
    display_name = name[:37] + "..." if len(name) > 40 else name
    
    print(f"{idx:<6} {topic_id:<10} {count:<12} {display_name:<40} {words_str}")


TOP 50 TOPICS WITH KEYWORDS

Total topics discovered (excluding outliers): 244
Documents assigned to topics: 26689
Outliers (unassigned): 33349

Rank   Topic ID   Articles     Topic Name                               Top Keywords
1      0          2307         0_england_league_euro_cup                england, league, euro, cup, manager, manchester, champions, club, ireland, boss
2      1          1597         1_gaza_israel_hamas_israeli              gaza, israel, hamas, israeli, palestinian, palestinians, netanyahu, hostages, war, aid
3      2          1299         2_electric_tesla_car_vehicles            electric, tesla, car, vehicles, vehicle, ev, cars, battery, auto, ford
4      3          1058         3_covid_19_vaccine_coronavirus           covid, 19, vaccine, coronavirus, pandemic, virus, vaccines, cases, variant, omicron
5      4          903          4_mortgage_housing_rates_home            mortgage, housing, rates, home, average, rate, homebuyers, market, estate, homes
6      

In [19]:
df_to_filter = df.copy()

# first set of words to filter out from headlines (case-insensitive)
final_words_to_filter = [
    'league', 'cup', 'manchester', 'champions', 'club', 'uaw', 'harris', 'kamala',
    'dnc', 'movie', 'oscar', 'fashion', 'beyoncé', 'carter', 'jimmy', 'hackman',
    'antitrust', 'social', 'airlines', 'boeing', 'airline', 'deals', 'cyber', 'patriots',
    'seahawks', 'dolphins', 'browns', 'roster', 'campaign', 'mcdonald', 'lobster',
    'abortion', 'abortions', 'mifepristone', 'ivf', 'khali', 'immigration', 'mahmoud',
    'prix', 'f1', 'verstappen', 'hamilton', 'ferrari', 'education', 'harvard', 'university',
    'niger', 'gabon', 'championships', 'medal', 'dog', 'dogs', 'retriever', 'puppy', 'cat', 'adorable'
    'nasa', 'telescope', 'asteroid', 'mars', 'space', 'astronomers', 'webb', 'spacecraft',
]

print("Final round of filtering...")
print(f"Current dataset size (from english_news_v2): {len(df_to_filter)} articles")

# Create a mask to identify articles with irrelevant headlines
# Check if title contains any of the final filter words (case-insensitive)
filter_mask_v2 = df_to_filter['title'].str.lower().str.contains('|'.join(final_words_to_filter), na=False, regex=True)

# Count articles to be removed
articles_to_remove_v2 = filter_mask_v2.sum()
print(f"Articles to remove in this round: {articles_to_remove_v2} ({articles_to_remove_v2/len(df_to_filter)*100:.2f}%)")

# Filter the dataframe (keep articles that DON'T match the filter)
df_filtered_v2 = df_to_filter[~filter_mask_v2].copy()

print(f"Filtered dataset size (v2): {len(df_filtered_v2)} articles")
print(f"Total articles removed in this round: {len(df_to_filter) - len(df_filtered_v2)}")
print(f"Total articles removed from original: {len(df) - len(df_filtered_v2)} ({(len(df) - len(df_filtered_v2))/len(df)*100:.2f}%)")

# Save the final filtered dataset
output_filename = 'english_news_v6.csv'

# Save to CSV (already has the correct columns)
df_filtered_v2.to_csv(output_filename, index=False)


Final round of filtering...
Current dataset size (from english_news_v2): 82374 articles
Articles to remove in this round: 6443 (7.82%)
Filtered dataset size (v2): 75931 articles
Total articles removed in this round: 6443
Total articles removed from original: 6443 (7.82%)


# Third Topic Identification & Cleanup round

In [21]:
# Load the latest dataset
df = pd.read_csv('english_news_v6.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['publishedAt'].min()} to {df['publishedAt'].max()}")

# Prepare text data for topic modeling
# Combine title and description, remove missing values
df_topic = df.copy()
df_topic['combined_text'] = df_topic.apply(
    lambda row: f"{row['title']} {row['description']}" 
    if pd.notna(row['title']) and pd.notna(row['description']) 
    else (row['title'] if pd.notna(row['title']) else str(row['description']) if pd.notna(row['description']) else ''),
    axis=1
)

# Remove empty strings and very short texts
df_topic = df_topic[df_topic['combined_text'].str.strip() != ''].copy()
df_topic = df_topic[df_topic['combined_text'].str.len() > 20].copy()  # Minimum length

print(f"\nArticles prepared for topic modeling: {len(df_topic)}")
print(f"Average document length: {np.mean([len(doc) for doc in df_topic['combined_text']]):.1f} characters")
print(f"\nSample text (first 200 chars):")
print(df_topic['combined_text'].iloc[0][:200] if len(df_topic) > 0 else "No data")


Dataset shape: (75931, 5)
Columns: ['publishedAt', 'title', 'source', 'description', 'url']

Date range: 2020-11-01 16:52:50+00:00 to 2025-10-31 12:48:33+00:00

Articles prepared for topic modeling: 75930
Average document length: 217.3 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.


In [22]:
# Extract documents for topic modeling
documents = df_topic['combined_text'].tolist()

print(f"Total documents: {len(documents)}")

# Initialize BERTopic with optimized settings for news articles
print("\nInitializing BERTopic model...")
print("This may take a few minutes...")

# Use a lightweight embedding model for faster processing
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Configure dimensionality reduction and clustering
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Initialize BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=10,
    verbose=True,
    language='english'
)

# Fit the model
print("\nFitting the model to documents...")
topics, probs = topic_model.fit_transform(documents)

print(f"\nModel training completed!")
print(f"Number of topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
if -1 in topics:
    print(f"Outliers (topic -1): {topics.count(-1)} documents")


Total documents: 75930

Initializing BERTopic model...
This may take a few minutes...


2025-10-31 15:28:15,188 - BERTopic - Embedding - Transforming documents to embeddings.



Fitting the model to documents...


Batches: 100%|██████████| 2373/2373 [01:04<00:00, 36.87it/s]
2025-10-31 15:29:20,219 - BERTopic - Embedding - Completed ✓
2025-10-31 15:29:20,220 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-31 15:30:08,728 - BERTopic - Dimensionality - Completed ✓
2025-10-31 15:30:08,730 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR


Model training completed!
Number of topics discovered: 242
Outliers (topic -1): 31554 documents


In [24]:
# Get topic information and display TOP 50 TOPICS
topic_info = topic_model.get_topic_info()

print("=" * 80)
print("TOP 50 TOPICS WITH KEYWORDS")
print("=" * 80)

# Filter out outlier topic (-1) for display
topics_to_show = topic_info[topic_info['Topic'] != -1].head(50)

print(f"\nTotal topics discovered (excluding outliers): {len(topic_info[topic_info['Topic'] != -1])}")
print(f"Documents assigned to topics: {topics_to_show['Count'].sum()}")
if -1 in topics:
    outliers_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if len(topic_info[topic_info['Topic'] == -1]) > 0 else 0
    print(f"Outliers (unassigned): {outliers_count}")

print("\n" + "=" * 80)
print(f"{'Rank':<6} {'Topic ID':<10} {'Articles':<12} {'Topic Name':<40} {'Top Keywords'}")
print("=" * 80)

for idx, (_, row) in enumerate(topics_to_show.iterrows(), 1):
    topic_id = row['Topic']
    count = row['Count']
    name = row['Name']
    
    # Get top words for this topic
    top_words = topic_model.get_topic(topic_id)
    words_str = ", ".join([word for word, score in top_words[:10]])
    
    # Shorten topic name if too long
    display_name = name[:37] + "..." if len(name) > 40 else name
    
    print(f"{idx:<6} {topic_id:<10} {count:<12} {display_name:<40} {words_str}")


TOP 50 TOPICS WITH KEYWORDS

Total topics discovered (excluding outliers): 242
Documents assigned to topics: 23660
Outliers (unassigned): 31554

Rank   Topic ID   Articles     Topic Name                               Top Keywords
1      0          2167         0_england_league_euro_manager            england, league, euro, manager, cup, manchester, champions, ireland, win, scotland
2      1          1581         1_gaza_israel_hamas_israeli              gaza, israel, hamas, israeli, palestinian, netanyahu, palestinians, hostages, war, aid
3      2          1274         2_electric_tesla_car_vehicles            electric, tesla, car, vehicles, vehicle, ev, cars, auto, battery, ford
4      3          1117         3_covid_19_vaccine_coronavirus           covid, 19, vaccine, coronavirus, pandemic, virus, cases, vaccines, variant, omicron
5      4          836          4_mortgage_housing_rates_home            mortgage, housing, rates, home, average, rate, homebuyers, market, level, prices
6   

In [26]:
df_to_filter = df.copy()

# set of words to filter out from headlines (case-insensitive)
final_words_to_filter = [
    'league', 'cup', 'manchester', 'champions', 'club', 'uaw', 'harris', 'kamala',
    'dnc', 'movie', 'oscar', 'fashion', 'beyoncé', 'carter', 'jimmy', 'hackman',
    'antitrust', 'social', 'airlines', 'boeing', 'airline', 'deals', 'cyber', 'patriots',
    'seahawks', 'dolphins', 'browns', 'roster', 'campaign', 'mcdonald', 'lobster',
    'abortion', 'abortions', 'mifepristone', 'ivf', 'khali', 'immigration', 'mahmoud',
    'prix', 'f1', 'verstappen', 'hamilton', 'ferrari', 'education', 'harvard', 'university',
    'niger', 'gabon', 'championships', 'medal', 'dog', 'dogs', 'retriever', 'puppy', 'cat', 'adorable'
    'nasa', 'telescope', 'asteroid', 'mars', 'space', 'astronomers', 'webb', 'spacecraft',
]

print("Filtering...")
print(f"Current dataset size: {len(df_to_filter)} articles")

# Create a mask to identify articles with irrelevant headlines
# Check if title contains any of the final filter words (case-insensitive)
filter_mask_v3 = df_to_filter['description'].str.lower().str.contains('|'.join(final_words_to_filter), na=False, regex=True)

# Count articles to be removed
articles_to_remove_v3 = filter_mask_v3.sum()
print(f"Articles to remove in this round: {articles_to_remove_v3} ({articles_to_remove_v3/len(df_to_filter)*100:.2f}%)")

# Filter the dataframe (keep articles that DON'T match the filter)
df_filtered_v3 = df_to_filter[~filter_mask_v3].copy()

print(f"Filtered dataset size (v3): {len(df_filtered_v3)} articles")
print(f"Total articles removed in this round: {len(df_to_filter) - len(df_filtered_v3)}")
print(f"Total articles removed from original: {len(df) - len(df_filtered_v3)} ({(len(df) - len(df_filtered_v3))/len(df)*100:.2f}%)")

# Save the final filtered dataset
output_filename = 'english_news_v7.csv'

# Save to CSV (already has the correct columns)
df_filtered_v3.to_csv(output_filename, index=False)


Filtering...
Current dataset size: 75931 articles
Articles to remove in this round: 7015 (9.24%)
Filtered dataset size (v3): 68916 articles
Total articles removed in this round: 7015
Total articles removed from original: 7015 (9.24%)


# Fourth Topic Identification & Cleanup round

In [27]:
# Load the latest dataset
df = pd.read_csv('english_news_v7.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['publishedAt'].min()} to {df['publishedAt'].max()}")

# Prepare text data for topic modeling
# Combine title and description, remove missing values
df_topic = df.copy()
df_topic['combined_text'] = df_topic.apply(
    lambda row: f"{row['title']} {row['description']}" 
    if pd.notna(row['title']) and pd.notna(row['description']) 
    else (row['title'] if pd.notna(row['title']) else str(row['description']) if pd.notna(row['description']) else ''),
    axis=1
)

# Remove empty strings and very short texts
df_topic = df_topic[df_topic['combined_text'].str.strip() != ''].copy()
df_topic = df_topic[df_topic['combined_text'].str.len() > 20].copy()  # Minimum length

print(f"\nArticles prepared for topic modeling: {len(df_topic)}")
print(f"Average document length: {np.mean([len(doc) for doc in df_topic['combined_text']]):.1f} characters")
print(f"\nSample text (first 200 chars):")
print(df_topic['combined_text'].iloc[0][:200] if len(df_topic) > 0 else "No data")


Dataset shape: (68916, 5)
Columns: ['publishedAt', 'title', 'source', 'description', 'url']

Date range: 2020-11-01 16:52:50+00:00 to 2025-10-31 12:48:33+00:00

Articles prepared for topic modeling: 68915
Average document length: 216.1 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.


In [28]:
# Extract documents for topic modeling
documents = df_topic['combined_text'].tolist()

print(f"Total documents: {len(documents)}")

# Initialize BERTopic with optimized settings for news articles
print("\nInitializing BERTopic model...")
print("This may take a few minutes...")

# Use a lightweight embedding model for faster processing
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Configure dimensionality reduction and clustering
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Initialize BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=10,
    verbose=True,
    language='english'
)

# Fit the model
print("\nFitting the model to documents...")
topics, probs = topic_model.fit_transform(documents)

print(f"\nModel training completed!")
print(f"Number of topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
if -1 in topics:
    print(f"Outliers (topic -1): {topics.count(-1)} documents")


Total documents: 68915

Initializing BERTopic model...
This may take a few minutes...


2025-10-31 15:38:24,071 - BERTopic - Embedding - Transforming documents to embeddings.



Fitting the model to documents...


Batches: 100%|██████████| 2154/2154 [00:57<00:00, 37.64it/s]
2025-10-31 15:39:21,906 - BERTopic - Embedding - Completed ✓
2025-10-31 15:39:21,907 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-31 15:40:03,572 - BERTopic - Dimensionality - Completed ✓
2025-10-31 15:40:03,574 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR


Model training completed!
Number of topics discovered: 232
Outliers (topic -1): 30525 documents


In [29]:
# Get topic information and display TOP 50 TOPICS
topic_info = topic_model.get_topic_info()

print("=" * 80)
print("TOP 50 TOPICS WITH KEYWORDS")
print("=" * 80)

# Filter out outlier topic (-1) for display
topics_to_show = topic_info[topic_info['Topic'] != -1].head(50)

print(f"\nTotal topics discovered (excluding outliers): {len(topic_info[topic_info['Topic'] != -1])}")
print(f"Documents assigned to topics: {topics_to_show['Count'].sum()}")
if -1 in topics:
    outliers_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if len(topic_info[topic_info['Topic'] == -1]) > 0 else 0
    print(f"Outliers (unassigned): {outliers_count}")

print("\n" + "=" * 80)
print(f"{'Rank':<6} {'Topic ID':<10} {'Articles':<12} {'Topic Name':<40} {'Top Keywords'}")
print("=" * 80)

for idx, (_, row) in enumerate(topics_to_show.iterrows(), 1):
    topic_id = row['Topic']
    count = row['Count']
    name = row['Name']
    
    # Get top words for this topic
    top_words = topic_model.get_topic(topic_id)
    words_str = ", ".join([word for word, score in top_words[:10]])
    
    # Shorten topic name if too long
    display_name = name[:37] + "..." if len(name) > 40 else name
    
    print(f"{idx:<6} {topic_id:<10} {count:<12} {display_name:<40} {words_str}")


TOP 50 TOPICS WITH KEYWORDS

Total topics discovered (excluding outliers): 232
Documents assigned to topics: 19734
Outliers (unassigned): 30525

Rank   Topic ID   Articles     Topic Name                               Top Keywords
1      0          1448         0_gaza_israel_hamas_israeli              gaza, israel, hamas, israeli, netanyahu, palestinian, palestinians, hostages, aid, rafah
2      1          1219         1_electric_tesla_car_vehicles            electric, tesla, car, vehicles, vehicle, ev, cars, auto, battery, ford
3      2          1099         2_england_euro_manager_ireland           england, euro, manager, ireland, scotland, rugby, premiership, chelsea, football, side
4      3          764          3_mortgage_housing_rates_home            mortgage, housing, rates, home, average, rate, homebuyers, level, estate, market
5      4          724          4_snow_tornadoes_storm_storms            snow, tornadoes, storm, storms, weather, tornado, severe, winter, midwest, rain
6 

In [30]:
df_to_filter = df.copy()

# set of words to filter out from headlines (case-insensitive)
final_words_to_filter = [
    'rugby', 'premiership', 'football', 'mortgage', 'housing', 'homebuyer', 'estate',
    'mortgages', 'tornadoes', 'snow', 'storms', 'midwest', 'weather', 'korea', 'korean',
    'missile', 'heat', 'temparature', 'temparatures', 'summer', 'heatwave', 'strike', 'workers',
    'nurses', 'doctors', 'classified', 'debate', 'union', 'senate', 'beautiful', 'megabill',
    'kendrick', 'lamar', 'concert', 'tv', 'lotus', 'songs', 'colorado', 'drought', 'lake', 'arizona',
    'ravens', 'titans', '49ers', 'chargers', 'wildfire', 'wildfires', 'firefighters', 'angeles',
    'college', 'ohio', 'coach', 'kissinger', 'schultz', 'helicopter', 'pilot', 'ipo', 'adani',
    'ipos', 'dance', 'dad', 'mom', 'baby', 'rfk', 'haley', 'nikki',
]

print("Filtering...")
print(f"Current dataset size: {len(df_to_filter)} articles")

# Create a mask to identify articles with irrelevant headlines
# Check if title contains any of the final filter words (case-insensitive)
filter_mask_v4 = df_to_filter['description'].str.lower().str.contains('|'.join(final_words_to_filter), na=False, regex=True)

# Count articles to be removed
articles_to_remove_v4 = filter_mask_v4.sum()
print(f"Articles to remove in this round: {articles_to_remove_v4} ({articles_to_remove_v4/len(df_to_filter)*100:.2f}%)")

# Filter the dataframe (keep articles that DON'T match the filter)
df_filtered_v4 = df_to_filter[~filter_mask_v4].copy()

print(f"Filtered dataset size (v4): {len(df_filtered_v4)} articles")
print(f"Total articles removed in this round: {len(df_to_filter) - len(df_filtered_v4)}")
print(f"Total articles removed from original: {len(df) - len(df_filtered_v4)} ({(len(df) - len(df_filtered_v4))/len(df)*100:.2f}%)")

# Save the final filtered dataset
output_filename = 'english_news_v8.csv'

# Save to CSV (already has the correct columns)
df_filtered_v4.to_csv(output_filename, index=False)


Filtering...
Current dataset size: 68916 articles
Articles to remove in this round: 12854 (18.65%)
Filtered dataset size (v4): 56062 articles
Total articles removed in this round: 12854
Total articles removed from original: 12854 (18.65%)


# Fifth Topic Identification & Cleanup round

In [31]:
# Load the latest dataset
df = pd.read_csv('english_news_v8.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['publishedAt'].min()} to {df['publishedAt'].max()}")

# Prepare text data for topic modeling
# Combine title and description, remove missing values
df_topic = df.copy()
df_topic['combined_text'] = df_topic.apply(
    lambda row: f"{row['title']} {row['description']}" 
    if pd.notna(row['title']) and pd.notna(row['description']) 
    else (row['title'] if pd.notna(row['title']) else str(row['description']) if pd.notna(row['description']) else ''),
    axis=1
)

# Remove empty strings and very short texts
df_topic = df_topic[df_topic['combined_text'].str.strip() != ''].copy()
df_topic = df_topic[df_topic['combined_text'].str.len() > 20].copy()  # Minimum length

print(f"\nArticles prepared for topic modeling: {len(df_topic)}")
print(f"Average document length: {np.mean([len(doc) for doc in df_topic['combined_text']]):.1f} characters")
print(f"\nSample text (first 200 chars):")
print(df_topic['combined_text'].iloc[0][:200] if len(df_topic) > 0 else "No data")


Dataset shape: (56062, 5)
Columns: ['publishedAt', 'title', 'source', 'description', 'url']

Date range: 2020-11-01 18:18:06+00:00 to 2025-10-31 12:48:33+00:00

Articles prepared for topic modeling: 56061
Average document length: 212.4 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.


In [32]:
# Extract documents for topic modeling
documents = df_topic['combined_text'].tolist()

print(f"Total documents: {len(documents)}")

# Initialize BERTopic with optimized settings for news articles
print("\nInitializing BERTopic model...")
print("This may take a few minutes...")

# Use a lightweight embedding model for faster processing
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Configure dimensionality reduction and clustering
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Initialize BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=10,
    verbose=True,
    language='english'
)

# Fit the model
print("\nFitting the model to documents...")
topics, probs = topic_model.fit_transform(documents)

print(f"\nModel training completed!")
print(f"Number of topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
if -1 in topics:
    print(f"Outliers (topic -1): {topics.count(-1)} documents")


Total documents: 56061

Initializing BERTopic model...
This may take a few minutes...


2025-10-31 16:18:05,789 - BERTopic - Embedding - Transforming documents to embeddings.



Fitting the model to documents...


Batches: 100%|██████████| 1752/1752 [00:46<00:00, 37.52it/s]
2025-10-31 16:18:53,376 - BERTopic - Embedding - Completed ✓
2025-10-31 16:18:53,377 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-31 16:19:25,813 - BERTopic - Dimensionality - Completed ✓
2025-10-31 16:19:25,815 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR


Model training completed!
Number of topics discovered: 205
Outliers (topic -1): 25295 documents


In [33]:
# Get topic information and display TOP 50 TOPICS
topic_info = topic_model.get_topic_info()

print("=" * 80)
print("TOP 50 TOPICS WITH KEYWORDS")
print("=" * 80)

# Filter out outlier topic (-1) for display
topics_to_show = topic_info[topic_info['Topic'] != -1].head(50)

print(f"\nTotal topics discovered (excluding outliers): {len(topic_info[topic_info['Topic'] != -1])}")
print(f"Documents assigned to topics: {topics_to_show['Count'].sum()}")
if -1 in topics:
    outliers_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if len(topic_info[topic_info['Topic'] == -1]) > 0 else 0
    print(f"Outliers (unassigned): {outliers_count}")

print("\n" + "=" * 80)
print(f"{'Rank':<6} {'Topic ID':<10} {'Articles':<12} {'Topic Name':<40} {'Top Keywords'}")
print("=" * 80)

for idx, (_, row) in enumerate(topics_to_show.iterrows(), 1):
    topic_id = row['Topic']
    count = row['Count']
    name = row['Name']
    
    # Get top words for this topic
    top_words = topic_model.get_topic(topic_id)
    words_str = ", ".join([word for word, score in top_words[:10]])
    
    # Shorten topic name if too long
    display_name = name[:37] + "..." if len(name) > 40 else name
    
    print(f"{idx:<6} {topic_id:<10} {count:<12} {display_name:<40} {words_str}")


TOP 50 TOPICS WITH KEYWORDS

Total topics discovered (excluding outliers): 205
Documents assigned to topics: 16682
Outliers (unassigned): 25295

Rank   Topic ID   Articles     Topic Name                               Top Keywords
1      0          1222         0_gaza_israel_hamas_israeli              gaza, israel, hamas, israeli, netanyahu, palestinian, palestinians, hostages, war, aid
2      1          1121         1_electric_tesla_car_vehicles            electric, tesla, car, vehicles, vehicle, ev, cars, auto, ford, evs
3      2          763          2_vs_players_game_bowl                   vs, players, game, bowl, kraken, season, nhl, basketball, super, arena
4      3          741          3_england_euro_manager_ireland           england, euro, manager, ireland, scotland, chelsea, side, boss, liverpool, season
5      4          544          4_hunter_court_trial_trump               hunter, court, trial, trump, supreme, case, justice, biden, doj, impeachment
6      5          526     

In [34]:
df_to_filter = df.copy()

# set of words to filter out from headlines (case-insensitive)
final_words_to_filter = [
    'game', 'kraken', 'nhl', 'season', 'basketball', 'manager', 'chelsea', 'scotland', 'liverpool', 
    'impeachment', 'lawsuit', 'montana', 'meat', 'diet', 'beef', 'healthy', 'grocery', 'costco', 
    'retailer', 'mortgage', 'homes', 'renters', 'hurricane', 'storm', 'king', 'charles', 'prince', 
    'podcast', 'sweeny', 'radio'
]

print("Filtering...")
print(f"Current dataset size: {len(df_to_filter)} articles")

# Create a mask to identify articles with irrelevant headlines
# Check if title contains any of the final filter words (case-insensitive)
filter_mask_v5 = df_to_filter['description'].str.lower().str.contains('|'.join(final_words_to_filter), na=False, regex=True)

# Count articles to be removed
articles_to_remove_v5 = filter_mask_v5.sum()
print(f"Articles to remove in this round: {articles_to_remove_v5} ({articles_to_remove_v5/len(df_to_filter)*100:.2f}%)")

# Filter the dataframe (keep articles that DON'T match the filter)
df_filtered_v5 = df_to_filter[~filter_mask_v5].copy()

print(f"Filtered dataset size (v5): {len(df_filtered_v5)} articles")
print(f"Total articles removed in this round: {len(df_to_filter) - len(df_filtered_v5)}")
print(f"Total articles removed from original: {len(df) - len(df_filtered_v5)} ({(len(df) - len(df_filtered_v5))/len(df)*100:.2f}%)")

# Save the final filtered dataset
output_filename = 'english_news_v9.csv'

# Save to CSV (already has the correct columns)
df_filtered_v5.to_csv(output_filename, index=False)


Filtering...
Current dataset size: 56062 articles
Articles to remove in this round: 7735 (13.80%)
Filtered dataset size (v5): 48327 articles
Total articles removed in this round: 7735
Total articles removed from original: 7735 (13.80%)


# Sanity Check

In [35]:
candidate_labels = [
    "electricity or energy consumption",
    "electricity or energy production",
    "financial markets",
    "geopolitical news",
    "weather",
    "sports",
    "entertainment",
    "other"
]

hypothesis_template = "This text is about {}."

In [36]:
import torch
from transformers import pipeline


# Check for device availability in order: CUDA -> MPS -> CPU
if torch.cuda.is_available():
    device = 0
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = -1

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-base-zeroshot-v2.0",
    device=device
)

def classify_batch(texts, labels, hypothesis_template, batch_size=16):
    """Classify a batch of texts using zero-shot classification."""
    valid_texts = []
    valid_indices = []
    
    for idx, text in enumerate(texts):
        if pd.notna(text) and text.strip() != '':
            valid_texts.append(text)
            valid_indices.append(idx)
    
    if not valid_texts:
        return {}, {}
    
    results = classifier(
        valid_texts,
        labels,
        hypothesis_template=hypothesis_template,
        multi_label=False,
        batch_size=batch_size
    )
    
    classifications_dict = {}
    scores_dict = {}
    
    for i, (idx, result) in enumerate(zip(valid_indices, results)):
        classifications_dict[idx] = result['labels'][0]
        scores_dict[idx] = result['scores'][0]
    
    for idx in range(len(texts)):
        if idx not in classifications_dict:
            classifications_dict[idx] = None
            scores_dict[idx] = 0.0
    
    return classifications_dict, scores_dict

latest_news_df = pd.read_csv('english_news_v9.csv')

titles = latest_news_df['title'].tolist()
classifications_dict, scores_dict = classify_batch(
    titles, candidate_labels, hypothesis_template, batch_size=16
)

latest_news_df['classification'] = [classifications_dict[i] for i in range(len(latest_news_df))]
latest_news_df['classification_score'] = [scores_dict[i] for i in range(len(latest_news_df))]

other_mask = latest_news_df['classification'] == "other (not related to these energy price drivers)"
num_other = other_mask.sum()

if num_other > 0:
    other_indices = latest_news_df[other_mask].index
    descriptions = latest_news_df.loc[other_indices, 'description'].tolist()
    
    other_classifications_dict, other_scores_dict = classify_batch(
        descriptions, candidate_labels, hypothesis_template, batch_size=16
    )
    
    for i, idx in enumerate(other_indices):
        latest_news_df.loc[idx, 'classification'] = other_classifications_dict[i]
        latest_news_df.loc[idx, 'classification_score'] = other_scores_dict[i]

final_other = (latest_news_df['classification'] == "other (not related to these energy price drivers)").sum()

print(f"Classification completed: {len(latest_news_df)} articles processed")
print(f"Articles classified as 'other': {final_other} ({final_other/len(latest_news_df)*100:.1f}%)")
print(f"\nClassification distribution:")
print(latest_news_df['classification'].value_counts())
print(f"\nAverage score: {latest_news_df['classification_score'].mean():.3f}")
print(f"Median score: {latest_news_df['classification_score'].median():.3f}")

Device set to use mps


Classification completed: 48327 articles processed
Articles classified as 'other': 0 (0.0%)

Classification distribution:
classification
geopolitical news                    25307
financial markets                     8574
other                                 5657
entertainment                         3408
sports                                2071
weather                               1532
electricity or energy production      1421
electricity or energy consumption      356
Name: count, dtype: int64

Average score: 0.774
Median score: 0.883


# Sixth Topic Identification & Cleanup round

In [2]:
# Load the latest dataset
df = pd.read_csv('english_news_v9.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['publishedAt'].min()} to {df['publishedAt'].max()}")

# Prepare text data for topic modeling
# Combine title and description, remove missing values
df_topic = df.copy()
df_topic['combined_text'] = df_topic.apply(
    lambda row: f"{row['title']} {row['description']}" 
    if pd.notna(row['title']) and pd.notna(row['description']) 
    else (row['title'] if pd.notna(row['title']) else str(row['description']) if pd.notna(row['description']) else ''),
    axis=1
)

# Remove empty strings and very short texts
df_topic = df_topic[df_topic['combined_text'].str.strip() != ''].copy()
df_topic = df_topic[df_topic['combined_text'].str.len() > 20].copy()  # Minimum length

print(f"\nArticles prepared for topic modeling: {len(df_topic)}")
print(f"Average document length: {np.mean([len(doc) for doc in df_topic['combined_text']]):.1f} characters")
print(f"\nSample text (first 200 chars):")
print(df_topic['combined_text'].iloc[0][:200] if len(df_topic) > 0 else "No data")

Dataset shape: (48327, 5)
Columns: ['publishedAt', 'title', 'source', 'description', 'url']

Date range: 2020-11-01 18:18:06+00:00 to 2025-10-31 12:48:33+00:00

Articles prepared for topic modeling: 48326
Average document length: 210.7 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.

Articles prepared for topic modeling: 48326
Average document length: 210.7 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.


In [3]:
# Extract documents for topic modeling
documents = df_topic['combined_text'].tolist()

print(f"Total documents: {len(documents)}")

# Initialize BERTopic with optimized settings for news articles
print("\nInitializing BERTopic model...")
print("This may take a few minutes...")

# Use a lightweight embedding model for faster processing
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Configure dimensionality reduction and clustering
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Initialize BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=10,
    verbose=True,
    language='english'
)

# Fit the model
print("\nFitting the model to documents...")
topics, probs = topic_model.fit_transform(documents)

print(f"\nModel training completed!")
print(f"Number of topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
if -1 in topics:
    print(f"Outliers (topic -1): {topics.count(-1)} documents")

Total documents: 48326

Initializing BERTopic model...
This may take a few minutes...


2025-11-02 11:57:36,500 - BERTopic - Embedding - Transforming documents to embeddings.



Fitting the model to documents...


Batches: 100%|██████████| 1511/1511 [01:52<00:00, 13.39it/s]

2025-11-02 11:59:29,775 - BERTopic - Embedding - Completed ✓
2025-11-02 11:59:29,776 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-02 11:59:29,775 - BERTopic - Embedding - Completed ✓
2025-11-02 11:59:29,776 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-02 12:00:04,753 - BERTopic - Dimensionality - Completed ✓
2025-11-02 12:00:04,754 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-02 12:00:04,753 - BERTopic - Dimensionality - Completed ✓
2025-11-02 12:00:04,754 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-02 12:00:07,161 - BERTopic - Cluster - Completed ✓
2025-11-02 12:00:07,168 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-02 12:00:07,161 - BERTopic - Cluster - Completed ✓
2025-11-02 12:00:07,168 - BERTopic - Representation - Fine-tuning topics using represent


Model training completed!
Number of topics discovered: 165
Outliers (topic -1): 21579 documents


In [4]:
# Get topic information and display TOP 50 TOPICS
topic_info = topic_model.get_topic_info()

print("=" * 80)
print("TOP 50 TOPICS WITH KEYWORDS")
print("=" * 80)

# Filter out outlier topic (-1) for display
topics_to_show = topic_info[topic_info['Topic'] != -1].head(50)

print(f"\nTotal topics discovered (excluding outliers): {len(topic_info[topic_info['Topic'] != -1])}")
print(f"Documents assigned to topics: {topics_to_show['Count'].sum()}")
if -1 in topics:
    outliers_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if len(topic_info[topic_info['Topic'] == -1]) > 0 else 0
    print(f"Outliers (unassigned): {outliers_count}")

print("\n" + "=" * 80)
print(f"{'Rank':<6} {'Topic ID':<10} {'Articles':<12} {'Topic Name':<40} {'Top Keywords'}")
print("=" * 80)

for idx, (_, row) in enumerate(topics_to_show.iterrows(), 1):
    topic_id = row['Topic']
    count = row['Count']
    name = row['Name']
    
    # Get top words for this topic
    top_words = topic_model.get_topic(topic_id)
    words_str = ", ".join([word for word, score in top_words[:10]])
    
    # Shorten topic name if too long
    display_name = name[:37] + "..." if len(name) > 40 else name
    
    print(f"{idx:<6} {topic_id:<10} {count:<12} {display_name:<40} {words_str}")

TOP 50 TOPICS WITH KEYWORDS

Total topics discovered (excluding outliers): 165
Documents assigned to topics: 16299
Outliers (unassigned): 21579

Rank   Topic ID   Articles     Topic Name                               Top Keywords
1      0          1411         0_gaza_israel_hamas_israeli              gaza, israel, hamas, israeli, netanyahu, palestinian, palestinians, lebanon, war, hostages
2      1          826          1_covid_19_vaccine_coronavirus           covid, 19, vaccine, coronavirus, pandemic, virus, cases, vaccines, omicron, variant
3      2          640          2_zoo_watch_species_bear                 zoo, watch, species, bear, whale, shark, endangered, rescued, sea, spotted
4      3          593          3_her_watch_she_love                     her, watch, she, love, his, best, show, gma, story, and
5      4          563          4_execution_nitrogen_death_man           execution, nitrogen, death, man, alabama, police, inmate, prison, officer, officers
6      5          50

In [5]:
df_to_filter = df.copy()

# set of words to filter out from headlines (case-insensitive)
final_words_to_filter = [
    'zoo', 'bear', 'whale', 'shark', 'drugs', 'ozempic', 'diabetes', 'alzheimer',
    'lgbtq', 'marriage', 'museum', 'art', 'painting', 'ufc'


]

print("Filtering...")
print(f"Current dataset size: {len(df_to_filter)} articles")

# Create a mask to identify articles with irrelevant headlines
# Check if title contains any of the final filter words (case-insensitive)
filter_mask_v6 = df_to_filter['description'].str.lower().str.contains('|'.join(final_words_to_filter), na=False, regex=True)

# Count articles to be removed
articles_to_remove_v6 = filter_mask_v6.sum()
print(f"Articles to remove in this round: {articles_to_remove_v6} ({articles_to_remove_v6/len(df_to_filter)*100:.2f}%)")

# Filter the dataframe (keep articles that DON'T match the filter)
df_filtered_v6 = df_to_filter[~filter_mask_v6].copy()

print(f"Filtered dataset size (v6): {len(df_filtered_v6)} articles")
print(f"Total articles removed in this round: {len(df_to_filter) - len(df_filtered_v6)}")
print(f"Total articles removed from original: {len(df) - len(df_filtered_v6)} ({(len(df) - len(df_filtered_v6))/len(df)*100:.2f}%)")

# Save the final filtered dataset
output_filename = 'english_news_v10.csv'

# Save to CSV (already has the correct columns)
df_filtered_v6.to_csv(output_filename, index=False)

Filtering...
Current dataset size: 48327 articles
Articles to remove in this round: 6504 (13.46%)
Filtered dataset size (v6): 41823 articles
Total articles removed in this round: 6504
Total articles removed from original: 6504 (13.46%)


# Seventh Topic Identification & Cleanup round

In [6]:
# Load the latest dataset
df = pd.read_csv('english_news_v10.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['publishedAt'].min()} to {df['publishedAt'].max()}")

# Prepare text data for topic modeling
# Combine title and description, remove missing values
df_topic = df.copy()
df_topic['combined_text'] = df_topic.apply(
    lambda row: f"{row['title']} {row['description']}" 
    if pd.notna(row['title']) and pd.notna(row['description']) 
    else (row['title'] if pd.notna(row['title']) else str(row['description']) if pd.notna(row['description']) else ''),
    axis=1
)

# Remove empty strings and very short texts
df_topic = df_topic[df_topic['combined_text'].str.strip() != ''].copy()
df_topic = df_topic[df_topic['combined_text'].str.len() > 20].copy()  # Minimum length

print(f"\nArticles prepared for topic modeling: {len(df_topic)}")
print(f"Average document length: {np.mean([len(doc) for doc in df_topic['combined_text']]):.1f} characters")
print(f"\nSample text (first 200 chars):")
print(df_topic['combined_text'].iloc[0][:200] if len(df_topic) > 0 else "No data")

Dataset shape: (41823, 5)
Columns: ['publishedAt', 'title', 'source', 'description', 'url']

Date range: 2020-11-01 18:18:06+00:00 to 2025-10-31 12:45:04+00:00

Articles prepared for topic modeling: 41822
Average document length: 207.4 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.

Articles prepared for topic modeling: 41822
Average document length: 207.4 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.


In [7]:
# Extract documents for topic modeling
documents = df_topic['combined_text'].tolist()

print(f"Total documents: {len(documents)}")

# Initialize BERTopic with optimized settings for news articles
print("\nInitializing BERTopic model...")
print("This may take a few minutes...")

# Use a lightweight embedding model for faster processing
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Configure dimensionality reduction and clustering
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Initialize BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=10,
    verbose=True,
    language='english'
)

# Fit the model
print("\nFitting the model to documents...")
topics, probs = topic_model.fit_transform(documents)

print(f"\nModel training completed!")
print(f"Number of topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
if -1 in topics:
    print(f"Outliers (topic -1): {topics.count(-1)} documents")

Total documents: 41822

Initializing BERTopic model...
This may take a few minutes...


2025-11-02 13:04:44,082 - BERTopic - Embedding - Transforming documents to embeddings.



Fitting the model to documents...


Batches: 100%|██████████| 1307/1307 [01:41<00:00, 12.90it/s]
2025-11-02 13:06:25,716 - BERTopic - Embedding - Completed ✓
2025-11-02 13:06:25,717 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-02 13:06:45,083 - BERTopic - Dimensionality - Completed ✓
2025-11-02 13:06:45,084 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-02 13:06:47,359 - BERTopic - Cluster - Completed ✓
2025-11-02 13:06:47,364 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-02 13:06:48,004 - BERTopic - Representation - Completed ✓



Model training completed!
Number of topics discovered: 127
Outliers (topic -1): 19302 documents


In [8]:
# Get topic information and display TOP 50 TOPICS
topic_info = topic_model.get_topic_info()

print("=" * 80)
print("TOP 50 TOPICS WITH KEYWORDS")
print("=" * 80)

# Filter out outlier topic (-1) for display
topics_to_show = topic_info[topic_info['Topic'] != -1].head(50)

print(f"\nTotal topics discovered (excluding outliers): {len(topic_info[topic_info['Topic'] != -1])}")
print(f"Documents assigned to topics: {topics_to_show['Count'].sum()}")
if -1 in topics:
    outliers_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if len(topic_info[topic_info['Topic'] == -1]) > 0 else 0
    print(f"Outliers (unassigned): {outliers_count}")

print("\n" + "=" * 80)
print(f"{'Rank':<6} {'Topic ID':<10} {'Articles':<12} {'Topic Name':<40} {'Top Keywords'}")
print("=" * 80)

for idx, (_, row) in enumerate(topics_to_show.iterrows(), 1):
    topic_id = row['Topic']
    count = row['Count']
    name = row['Name']
    
    # Get top words for this topic
    top_words = topic_model.get_topic(topic_id)
    words_str = ", ".join([word for word, score in top_words[:10]])
    
    # Shorten topic name if too long
    display_name = name[:37] + "..." if len(name) > 40 else name
    
    print(f"{idx:<6} {topic_id:<10} {count:<12} {display_name:<40} {words_str}")

TOP 50 TOPICS WITH KEYWORDS

Total topics discovered (excluding outliers): 127
Documents assigned to topics: 15908
Outliers (unassigned): 19302

Rank   Topic ID   Articles     Topic Name                               Top Keywords
1      0          1315         0_gaza_israel_hamas_israeli              gaza, israel, hamas, israeli, palestinian, netanyahu, palestinians, lebanon, war, hostages
2      1          1056         1_disney_her_watch_you                   disney, her, watch, you, best, and, she, disneyland, the, year
3      2          906          2_england_title_tour_open                england, title, tour, open, sport, win, ireland, champion, euro, wins
4      3          803          3_covid_19_vaccine_coronavirus           covid, 19, vaccine, coronavirus, pandemic, virus, vaccines, cases, variant, omicron
5      4          508          4_interest_rates_fed_rate                interest, rates, fed, rate, reserve, powell, federal, bank, jerome, central
6      5          485     

In [9]:
df_to_filter = df.copy()

# set of words to filter out from headlines (case-insensitive)
final_words_to_filter = [
    'disney', 'disneyland', 'espn', 'jets', 'maduro', 'police', 'sudan', 'congo', 'ethiopia'
]

print("Filtering...")
print(f"Current dataset size: {len(df_to_filter)} articles")

# Create a mask to identify articles with irrelevant headlines
# Check if title contains any of the final filter words (case-insensitive)
filter_mask_v7 = df_to_filter['description'].str.lower().str.contains('|'.join(final_words_to_filter), na=False, regex=True)

# Count articles to be removed
articles_to_remove_v7 = filter_mask_v7.sum()
print(f"Articles to remove in this round: {articles_to_remove_v7} ({articles_to_remove_v7/len(df_to_filter)*100:.2f}%)")

# Filter the dataframe (keep articles that DON'T match the filter)
df_filtered_v7 = df_to_filter[~filter_mask_v7].copy()

print(f"Filtered dataset size (v7): {len(df_filtered_v7)} articles")
print(f"Total articles removed in this round: {len(df_to_filter) - len(df_filtered_v7)}")
print(f"Total articles removed from original: {len(df) - len(df_filtered_v7)} ({(len(df) - len(df_filtered_v7))/len(df)*100:.2f}%)")

# Save the final filtered dataset
output_filename = 'english_news_v11.csv'

# Save to CSV (already has the correct columns)
df_filtered_v7.to_csv(output_filename, index=False)

Filtering...
Current dataset size: 41823 articles
Articles to remove in this round: 1048 (2.51%)
Filtered dataset size (v7): 40775 articles
Total articles removed in this round: 1048
Total articles removed from original: 1048 (2.51%)


# 8th Topic Identification and Cleanup Round


In [10]:
# Load the latest dataset
df = pd.read_csv('english_news_v11.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['publishedAt'].min()} to {df['publishedAt'].max()}")

# Prepare text data for topic modeling
# Combine title and description, remove missing values
df_topic = df.copy()
df_topic['combined_text'] = df_topic.apply(
    lambda row: f"{row['title']} {row['description']}" 
    if pd.notna(row['title']) and pd.notna(row['description']) 
    else (row['title'] if pd.notna(row['title']) else str(row['description']) if pd.notna(row['description']) else ''),
    axis=1
)

# Remove empty strings and very short texts
df_topic = df_topic[df_topic['combined_text'].str.strip() != ''].copy()
df_topic = df_topic[df_topic['combined_text'].str.len() > 20].copy()  # Minimum length

print(f"\nArticles prepared for topic modeling: {len(df_topic)}")
print(f"Average document length: {np.mean([len(doc) for doc in df_topic['combined_text']]):.1f} characters")
print(f"\nSample text (first 200 chars):")
print(df_topic['combined_text'].iloc[0][:200] if len(df_topic) > 0 else "No data")

Dataset shape: (40775, 5)
Columns: ['publishedAt', 'title', 'source', 'description', 'url']

Date range: 2020-11-01 18:18:06+00:00 to 2025-10-31 12:45:04+00:00

Articles prepared for topic modeling: 40774
Average document length: 206.9 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.

Articles prepared for topic modeling: 40774
Average document length: 206.9 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.


In [11]:
# Extract documents for topic modeling
documents = df_topic['combined_text'].tolist()

print(f"Total documents: {len(documents)}")

# Initialize BERTopic with optimized settings for news articles
print("\nInitializing BERTopic model...")
print("This may take a few minutes...")

# Use a lightweight embedding model for faster processing
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Configure dimensionality reduction and clustering
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Initialize BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=10,
    verbose=True,
    language='english'
)

# Fit the model
print("\nFitting the model to documents...")
topics, probs = topic_model.fit_transform(documents)

print(f"\nModel training completed!")
print(f"Number of topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
if -1 in topics:
    print(f"Outliers (topic -1): {topics.count(-1)} documents")

Total documents: 40774

Initializing BERTopic model...
This may take a few minutes...


2025-11-02 13:35:54,785 - BERTopic - Embedding - Transforming documents to embeddings.



Fitting the model to documents...


Batches: 100%|██████████| 1275/1275 [01:34<00:00, 13.48it/s]
2025-11-02 13:37:29,674 - BERTopic - Embedding - Completed ✓
2025-11-02 13:37:29,674 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm

2025-11-02 13:37:29,674 - BERTopic - Embedding - Completed ✓
2025-11-02 13:37:29,674 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-02 13:37:48,569 - BERTopic - Dimensionality - Completed ✓
2025-11-02 13:37:48,571 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-02 13:37:48,569 - BERTopic - Dimensionality - Completed ✓
2025-11-02 13:37:48,571 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-02 13:37:50,741 - BERTopic - Cluster - Completed ✓
2025-11-02 13:37:50,746 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-02 13:37:50,741 - BERTopic - Cluster - Completed ✓
2025-11-02 13:37:50,746 - BERTopic - Representation - Fine-tuning topics using represent


Model training completed!
Number of topics discovered: 125
Outliers (topic -1): 17086 documents


In [13]:
# Get topic information and display TOP 100 TOPICS
topic_info = topic_model.get_topic_info()

print("=" * 80)
print("TOP 100 TOPICS WITH KEYWORDS")
print("=" * 80)

# Filter out outlier topic (-1) for display
topics_to_show = topic_info[topic_info['Topic'] != -1].head(100)

print(f"\nTotal topics discovered (excluding outliers): {len(topic_info[topic_info['Topic'] != -1])}")
print(f"Documents assigned to topics: {topics_to_show['Count'].sum()}")
if -1 in topics:
    outliers_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if len(topic_info[topic_info['Topic'] == -1]) > 0 else 0
    print(f"Outliers (unassigned): {outliers_count}")

print("\n" + "=" * 80)
print(f"{'Rank':<6} {'Topic ID':<10} {'Articles':<12} {'Topic Name':<40} {'Top Keywords'}")
print("=" * 80)

for idx, (_, row) in enumerate(topics_to_show.iterrows(), 1):
    topic_id = row['Topic']
    count = row['Count']
    name = row['Name']
    
    # Get top words for this topic
    top_words = topic_model.get_topic(topic_id)
    words_str = ", ".join([word for word, score in top_words[:10]])
    
    # Shorten topic name if too long
    display_name = name[:37] + "..." if len(name) > 40 else name
    
    print(f"{idx:<6} {topic_id:<10} {count:<12} {display_name:<40} {words_str}")

TOP 100 TOPICS WITH KEYWORDS

Total topics discovered (excluding outliers): 125
Documents assigned to topics: 22197
Outliers (unassigned): 17086

Rank   Topic ID   Articles     Topic Name                               Top Keywords
1      0          1252         0_gaza_israel_hamas_israeli              gaza, israel, hamas, israeli, netanyahu, palestinian, palestinians, war, lebanon, hostages
2      1          923          1_england_tour_open_title                england, tour, open, title, win, sport, ireland, champion, euro, at
3      2          819          2_covid_19_vaccine_virus                 covid, 19, vaccine, virus, coronavirus, pandemic, cases, vaccines, variant, omicron
4      3          728          3_her_watch_she_year                     her, watch, she, year, the, time, and, best, with, his
5      4          676          4_electric_car_ev_vehicles               electric, car, ev, vehicles, vehicle, cars, auto, automakers, evs, battery
6      5          608          5_chi

In [14]:
df_to_filter = df.copy()

# set of words to filter out from headlines (case-insensitive)
final_words_to_filter = [
    'inmate', 'prison', 'chicken', 'meat', 'bourbon', 'fentanyl', 'drug', 'ocerdose',
    'prescription', 'pills', 'opioid', 'podcast', 'sweeney', 'jackpot', 'lottery',
    'powerball', 'lululemon', 'dyson', 'acne', 'skin', 'creams', 'cigarettes', 'vapes',
    'vaping', 'nicotine', 'marijuana', 'cannabis', 'weed', 'drug'
]

print("Filtering...")
print(f"Current dataset size: {len(df_to_filter)} articles")

# Create a mask to identify articles with irrelevant headlines
# Check if title contains any of the final filter words (case-insensitive)
filter_mask_v8 = df_to_filter['description'].str.lower().str.contains('|'.join(final_words_to_filter), na=False, regex=True)

# Count articles to be removed
articles_to_remove_v8 = filter_mask_v8.sum()
print(f"Articles to remove in this round: {articles_to_remove_v8} ({articles_to_remove_v8/len(df_to_filter)*100:.2f}%)")

# Filter the dataframe (keep articles that DON'T match the filter)
df_filtered_v8 = df_to_filter[~filter_mask_v8].copy()

print(f"Filtered dataset size (v8): {len(df_filtered_v8)} articles")
print(f"Total articles removed in this round: {len(df_to_filter) - len(df_filtered_v8)}")
print(f"Total articles removed from original: {len(df) - len(df_filtered_v8)} ({(len(df) - len(df_filtered_v8))/len(df)*100:.2f}%)")

# Save the final filtered dataset
output_filename = 'english_news_v12.csv'

# Save to CSV (already has the correct columns)
df_filtered_v8.to_csv(output_filename, index=False)

Filtering...
Current dataset size: 40775 articles
Articles to remove in this round: 964 (2.36%)
Filtered dataset size (v8): 39811 articles
Total articles removed in this round: 964
Total articles removed from original: 964 (2.36%)
Articles to remove in this round: 964 (2.36%)
Filtered dataset size (v8): 39811 articles
Total articles removed in this round: 964
Total articles removed from original: 964 (2.36%)


# 9th Topic Identification and Cleanup Round

In [15]:
# Load the latest dataset
df = pd.read_csv('english_news_v12.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['publishedAt'].min()} to {df['publishedAt'].max()}")

# Prepare text data for topic modeling
# Combine title and description, remove missing values
df_topic = df.copy()
df_topic['combined_text'] = df_topic.apply(
    lambda row: f"{row['title']} {row['description']}" 
    if pd.notna(row['title']) and pd.notna(row['description']) 
    else (row['title'] if pd.notna(row['title']) else str(row['description']) if pd.notna(row['description']) else ''),
    axis=1
)

# Remove empty strings and very short texts
df_topic = df_topic[df_topic['combined_text'].str.strip() != ''].copy()
df_topic = df_topic[df_topic['combined_text'].str.len() > 20].copy()  # Minimum length

print(f"\nArticles prepared for topic modeling: {len(df_topic)}")
print(f"Average document length: {np.mean([len(doc) for doc in df_topic['combined_text']]):.1f} characters")
print(f"\nSample text (first 200 chars):")
print(df_topic['combined_text'].iloc[0][:200] if len(df_topic) > 0 else "No data")

Dataset shape: (39811, 5)
Columns: ['publishedAt', 'title', 'source', 'description', 'url']

Date range: 2020-11-01 18:18:06+00:00 to 2025-10-31 12:45:04+00:00

Articles prepared for topic modeling: 39810
Average document length: 206.5 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.

Articles prepared for topic modeling: 39810
Average document length: 206.5 characters

Sample text (first 200 chars):
Ant Group: How it became a financial giant As China's Ant Group prepares for its stock market debut, we look at how it established its empire.


In [16]:
# Extract documents for topic modeling
documents = df_topic['combined_text'].tolist()

print(f"Total documents: {len(documents)}")

# Initialize BERTopic with optimized settings for news articles
print("\nInitializing BERTopic model...")
print("This may take a few minutes...")

# Use a lightweight embedding model for faster processing
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Configure dimensionality reduction and clustering
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Initialize BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=10,
    verbose=True,
    language='english'
)

# Fit the model
print("\nFitting the model to documents...")
topics, probs = topic_model.fit_transform(documents)

print(f"\nModel training completed!")
print(f"Number of topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
if -1 in topics:
    print(f"Outliers (topic -1): {topics.count(-1)} documents")

Total documents: 39810

Initializing BERTopic model...
This may take a few minutes...


2025-11-02 16:22:07,183 - BERTopic - Embedding - Transforming documents to embeddings.



Fitting the model to documents...


Batches: 100%|██████████| 1245/1245 [01:32<00:00, 13.43it/s]

2025-11-02 16:23:40,163 - BERTopic - Embedding - Completed ✓
2025-11-02 16:23:40,164 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-02 16:23:40,163 - BERTopic - Embedding - Completed ✓
2025-11-02 16:23:40,164 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-02 16:23:59,323 - BERTopic - Dimensionality - Completed ✓
2025-11-02 16:23:59,325 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-02 16:23:59,323 - BERTopic - Dimensionality - Completed ✓
2025-11-02 16:23:59,325 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-02 16:24:01,702 - BERTopic - Cluster - Completed ✓
2025-11-02 16:24:01,706 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-02 16:24:01,702 - BERTopic - Cluster - Completed ✓
2025-11-02 16:24:01,706 - BERTopic - Representation - Fine-tuning topics using represent


Model training completed!
Number of topics discovered: 115
Outliers (topic -1): 18066 documents


In [17]:
# Get topic information and display TOP 100 TOPICS
topic_info = topic_model.get_topic_info()

print("=" * 80)
print("TOP 100 TOPICS WITH KEYWORDS")
print("=" * 80)

# Filter out outlier topic (-1) for display
topics_to_show = topic_info[topic_info['Topic'] != -1].head(100)

print(f"\nTotal topics discovered (excluding outliers): {len(topic_info[topic_info['Topic'] != -1])}")
print(f"Documents assigned to topics: {topics_to_show['Count'].sum()}")
if -1 in topics:
    outliers_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if len(topic_info[topic_info['Topic'] == -1]) > 0 else 0
    print(f"Outliers (unassigned): {outliers_count}")

print("\n" + "=" * 80)
print(f"{'Rank':<6} {'Topic ID':<10} {'Articles':<12} {'Topic Name':<40} {'Top Keywords'}")
print("=" * 80)

for idx, (_, row) in enumerate(topics_to_show.iterrows(), 1):
    topic_id = row['Topic']
    count = row['Count']
    name = row['Name']
    
    # Get top words for this topic
    top_words = topic_model.get_topic(topic_id)
    words_str = ", ".join([word for word, score in top_words[:10]])
    
    # Shorten topic name if too long
    display_name = name[:37] + "..." if len(name) > 40 else name
    
    print(f"{idx:<6} {topic_id:<10} {count:<12} {display_name:<40} {words_str}")

TOP 100 TOPICS WITH KEYWORDS

Total topics discovered (excluding outliers): 115
Documents assigned to topics: 20894
Outliers (unassigned): 18066

Rank   Topic ID   Articles     Topic Name                               Top Keywords
1      0          1240         0_gaza_israel_hamas_israeli              gaza, israel, hamas, israeli, netanyahu, palestinian, lebanon, palestinians, war, aid
2      1          898          1_england_tour_open_title                england, tour, open, title, win, sport, ireland, champion, at, euro
3      2          759          2_covid_19_vaccine_pandemic              covid, 19, vaccine, pandemic, coronavirus, virus, cases, vaccines, variant, omicron
4      3          677          3_electric_car_ev_vehicles               electric, car, ev, vehicles, vehicle, cars, auto, automakers, battery, evs
5      4          649          4_her_watch_she_year                     her, watch, she, year, the, and, time, his, with, of
6      5          514          5_interest_r

In [18]:
df_to_filter = df.copy()

# set of words to filter out from headlines (case-insensitive)
final_words_to_filter = [
    'podcast', 'medicare', 'obamacare', 'medicaid'
]

print("Filtering...")
print(f"Current dataset size: {len(df_to_filter)} articles")

# Create a mask to identify articles with irrelevant headlines
# Check if title contains any of the final filter words (case-insensitive)
filter_mask_v9 = df_to_filter['description'].str.lower().str.contains('|'.join(final_words_to_filter), na=False, regex=True)

# Count articles to be removed
articles_to_remove_v9 = filter_mask_v9.sum()
print(f"Articles to remove in this round: {articles_to_remove_v9} ({articles_to_remove_v9/len(df_to_filter)*100:.2f}%)")

# Filter the dataframe (keep articles that DON'T match the filter)
df_filtered_v9 = df_to_filter[~filter_mask_v9].copy()

print(f"Filtered dataset size (v9): {len(df_filtered_v9)} articles")
print(f"Total articles removed in this round: {len(df_to_filter) - len(df_filtered_v9)}")
print(f"Total articles removed from original: {len(df) - len(df_filtered_v9)} ({(len(df) - len(df_filtered_v9))/len(df)*100:.2f}%)")

# Save the final filtered dataset
output_filename = 'english_news_v13.csv'

# Save to CSV (already has the correct columns)
df_filtered_v9.to_csv(output_filename, index=False)

Filtering...
Current dataset size: 39811 articles
Articles to remove in this round: 67 (0.17%)
Filtered dataset size (v9): 39744 articles
Total articles removed in this round: 67
Total articles removed from original: 67 (0.17%)
